In [ ]:
"""
env setup:
conda create -y --name tf1_model_export python=3.7
conda activate tf1_model_export
# note: gpu support is not necessary for tensorflow
pip install "tensorflow<2"
pip install "csbdeep[tf1]"
# also install stardist in this example
pip install "stardist[tf1]"
"""

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys
import numpy as np
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from glob import glob
from tqdm import tqdm
from tifffile import imread
from csbdeep.utils import Path, normalize

from stardist import fill_label_holes, random_label_cmap, calculate_extents, gputools_available
from stardist.matching import matching, matching_dataset
from stardist.models import Config2D, StarDist2D, StarDistData2D
from skimage.color import rgb2hed

np.random.seed(42)
lbl_cmap = random_label_cmap()

# Data

In [ ]:
# Get sorted lists of image and mask file paths
X = sorted(glob('training_data/images/*.tif'))
Y = sorted(glob('training_data/masks/*.tif'))

# Check that each image file has a matching mask file with the same filename
assert all(Path(x).name==Path(y).name for x,y in zip(X,Y))

In [ ]:
print(len(X), len(Y))

In [ ]:
# Read all image and mask files
X = list(map(imread,X))
Y = list(map(imread,Y))

# Detect the number of image channels
n_channel = 1 if X[0].ndim == 2 else X[0].shape[-1]
print(f"Number of channels = {n_channel} ")

In [ ]:
X_orig = X.copy()
Y_orig = Y.copy()

In [ ]:
# extract H and DAB channels and take the sum
def preprocess_ihc(img):
    # Deconvolution
    hed = rgb2hed(img)
    H = hed[:, :, 0]
    DAB = hed[:, :, 2]
    new_channel = H + DAB

    return new_channel.astype(np.float32)

X = [preprocess_ihc(x) for x in X]

In [ ]:
n_channel = 1 if X[0].ndim == 2 else X[0].shape[-1]
print(f"Number of channels now = {n_channel}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8,4))

axes[0].imshow(X[1].squeeze(), cmap='gray')
axes[0].set_title("Processed input")
axes[0].axis('off')

axes[1].imshow(X_orig[1].squeeze(), cmap='gray')
axes[1].set_title("Original input")
axes[1].axis('off')

plt.tight_layout()
plt.show()

Normalize images and fill small label holes.

In [ ]:
axis_norm = (0,1)   # normalize channels independently
if n_channel > 1:
    print("Normalizing image channels %s." % ('jointly' if axis_norm is None or 2 in axis_norm else 'independently'))
    sys.stdout.flush()

X = [normalize(x,1,99,axis=axis_norm) for x in tqdm(X)]
Y = [fill_label_holes(y) for y in tqdm(Y)]

Split into train and validation datasets.

In [ ]:
assert len(X) > 1, "not enough training data"
rng = np.random.RandomState(42)
ind = rng.permutation(len(X))
n_val = max(1, int(round(0.15 * len(ind))))
ind_train, ind_val = ind[:-n_val], ind[-n_val:]
X_val, Y_val = [X[i] for i in ind_val]  , [Y[i] for i in ind_val]
X_trn, Y_trn = [X[i] for i in ind_train], [Y[i] for i in ind_train] 
print('number of images: %3d' % len(X))
print('- training:       %3d' % len(X_trn))
print('- validation:     %3d' % len(X_val))

In [ ]:
def plot_img_label(img, lbl, img_title="image", lbl_title="label", **kwargs):
    fig, (ai,al) = plt.subplots(1,2, figsize=(12,5), gridspec_kw=dict(width_ratios=(1.25,1)))
    im = ai.imshow(img, cmap='gray', clim=(0,1))
    ai.set_title(img_title)    
    fig.colorbar(im, ax=ai)
    al.imshow(lbl, cmap=lbl_cmap)
    al.set_title(lbl_title)
    plt.tight_layout()

In [ ]:
i = min(10, len(X)-1)
img, lbl = X[i], Y[i]
assert img.ndim in (2,3)
img = img if (img.ndim==2 or img.shape[-1]==3) else img[...,0]
plot_img_label(img,lbl)
None;

# Configuration

A `StarDist2D` model is specified via a `Config2D` object.

In [ ]:
print(Config2D.__doc__)

In [ ]:
# 32 is a good default choice (see 1_data.ipynb)
n_rays = 32

# Use OpenCL-based computations for data generator during training (requires 'gputools')
use_gpu = False and gputools_available()

# Predict on subsampled grid for increased efficiency and larger field of view
grid = (2,2)

conf = Config2D (
    n_rays       = n_rays,
    grid         = grid,
    use_gpu      = use_gpu,
    n_channel_in = n_channel,
    train_learning_rate = 5e-5,
    train_batch_size=2,
)
print(conf)
vars(conf)

In [ ]:
if use_gpu:
    from csbdeep.utils.tf import limit_gpu_memory
    # adjust as necessary: limit GPU memory to be used by TensorFlow to leave some to OpenCL-based computations
    limit_gpu_memory(0.8)
    # alternatively, try this:
    # limit_gpu_memory(None, allow_growth=True)

In [ ]:
from stardist.models import StarDist2D
model = StarDist2D(conf, name='custom_trained_model', basedir='.')
model.export_TF()

Check if the neural network has a large enough field of view to see up to the boundary of most objects.

In [ ]:
median_size = calculate_extents(list(Y), np.median)
fov = np.array(model._axes_tile_overlap('YX'))
print(f"median object size:      {median_size}")
print(f"network field of view :  {fov}")
if any(median_size > fov):
    print("WARNING: median object size larger than field of view of the neural network.")

# Data Augmentation

In [ ]:
def random_fliprot(img, mask): 
    assert img.ndim >= mask.ndim
    axes = tuple(range(mask.ndim))
    perm = tuple(np.random.permutation(axes))
    img = img.transpose(perm + tuple(range(mask.ndim, img.ndim))) 
    mask = mask.transpose(perm) 
    for ax in axes: 
        if np.random.rand() > 0.5:
            img = np.flip(img, axis=ax)
            mask = np.flip(mask, axis=ax)
    return img, mask 

def random_intensity_change(img):
    img = img*np.random.uniform(0.6,2) + np.random.uniform(-0.2,0.2)
    return img


def augmenter(x, y):
    """Augmentation of a single input/label image pair.
    x is an input image
    y is the corresponding ground-truth label image
    """
    x, y = random_fliprot(x, y)
    x = random_intensity_change(x)
    # add some gaussian noise
    sig = 0.02*np.random.uniform(0,1)
    x = x + sig*np.random.normal(0,1,x.shape)
    return x, y

In [ ]:
# plot some augmented examples
img, lbl = X[10],Y[10]
plot_img_label(img, lbl)
for _ in range(3):
    img_aug, lbl_aug = augmenter(img,lbl)
    plot_img_label(img_aug, lbl_aug, img_title="image augmented", lbl_title="label augmented")

# Training

In [ ]:
training = model.train(X_trn, Y_trn, validation_data=(X_val,Y_val), augmenter=augmenter, 
            epochs=300, steps_per_epoch=150)

In [ ]:
# training.history

plt.figure(figsize=(8,5))

plt.plot(training.history['loss'], label='Train loss')
plt.plot(training.history['val_loss'], label='Val loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('StarDist Training Curve')
plt.legend()
plt.grid()
plt.show()

# Threshold optimization

While the default values for the probability and non-maximum suppression thresholds already yield good results in many cases, we still recommend to adapt the thresholds to your data. The optimized threshold values are saved to disk and will be automatically loaded with the model.

In [ ]:
model.optimize_thresholds(X_val, Y_val)

# Evaluation

First predict the labels for all validation images:

In [ ]:
Y_val_pred = [model.predict_instances(x, n_tiles=model._guess_n_tiles(x), show_tile_progress=False)[0]
              for x in tqdm(X_val)]

Plot a GT/prediction example 

In [ ]:
plot_img_label(X_val[2],Y_val[2], lbl_title="label GT")
plot_img_label(X_val[2],Y_val_pred[2], lbl_title="label Pred")

In [ ]:
idx = 2

img = X_val[idx]
pred_labels = Y_val_pred[idx]  
gt_labels = Y_val[idx]   

In [ ]:
gt = gt_labels
pred = pred_labels

# normalize to binary if needed
gt_bin = (gt > 0).astype(np.uint8)
pred_bin = (pred > 0).astype(np.uint8)

overlay = np.zeros((*gt.shape, 3), dtype=np.float32)

# Red = Ground Truth
overlay[..., 0] = gt_bin

# Green = Prediction
overlay[..., 1] = pred_bin

# Calculate overlap (yellow regions)
intersection = np.logical_and(gt_bin, pred_bin).sum()
union = np.logical_or(gt_bin, pred_bin).sum()

overlap_percent = (intersection / union * 100) if union > 0 else 0

# Plot
plt.figure(figsize=(6,6))
plt.imshow(overlay)

plt.title(
   f"IoU: {overlap_percent:.2f}%"
)

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='red', label='Ground Truth (GT)'),
    Patch(facecolor='lightgreen', label='Prediction'),
    Patch(facecolor='yellow', label='Overlap (GT ∩ Prediction)')
]

plt.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=3
)
plt.axis("off")
plt.show()

In [ ]:
model.export_TF(fname='custom_fluo_model')

In [ ]:
import shutil

src = "custom_fluo_model.zip"
dst = "/mnt/c/Users/19utk/Downloads/starDist_models/custom_fluo_model.zip"

shutil.copy(src, dst)

In [ ]:
# unzip custom_fluo_model and use in starDist script 